# Customer Churn Prediction Demo

Notebook de demo de negocio para predicción de churn usando el dataset **Telco Customer Churn**.

**Objetivo:** construir un flujo simple, claro y orientado a producción para explorar datos, entrenar modelos, registrar experimentos con MLflow y ejecutar una predicción de ejemplo.


# 0. Descarga del Dataset

Antes de iniciar el flujo principal, descargamos el archivo `telco_churn.csv` desde la fuente pÃºblica para que el notebook pueda ejecutarse incluso en una mÃ¡quina nueva.


In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())


# 1. Entorno y Dependencias

Este notebook está pensado para ejecutarse con **Python 3.11+** en **Jupyter Lab** dentro de un droplet de DigitalOcean usando **uv**.

## Instalación rápida con uv

```bash
uv venv
source .venv/bin/activate
uv pip install pandas numpy matplotlib seaborn scikit-learn mlflow jupyterlab
```

## Nota
- Asegúrate de que el archivo `telco_churn.csv` esté en el mismo directorio que este notebook.
- El notebook está dividido en pasos para facilitar la explicación durante la demo.


In [ ]:
# Paso 1: Importaciones y configuración general
from pathlib import Path
from io import StringIO

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import mlflow
import mlflow.sklearn

sns.set_theme(style='whitegrid', palette='Blues')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

DATA_PATH = Path('telco_churn.csv')
MLFLOW_TRACKING_URI = 'file:./mlruns'
EXPERIMENT_NAME = 'churn_prediction_demo'

print('Entorno listo para iniciar la demo.')


# 2. Carga de Datos

En esta sección cargamos el dataset y hacemos una revisión rápida de estructura, tamaño y valores faltantes.


In [ ]:
# Paso 2: Cargar dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(f'No se encontró el archivo: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
print(f'Dataset cargado correctamente desde: {DATA_PATH.resolve()}')


In [ ]:
# Vista inicial
df.head()


In [ ]:
# Tamaño del dataset
print(f'Shape del dataset: {df.shape}')


In [ ]:
# Resumen técnico con info()
buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())


In [ ]:
# Resumen de valores faltantes
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values(by='missing_count', ascending=False)

missing_summary[missing_summary['missing_count'] > 0] if missing_summary['missing_count'].sum() > 0 else pd.DataFrame({'message': ['No se detectaron valores faltantes nativos antes de limpieza.']})


# 3. Limpieza de Datos

Aquí convertimos columnas, tratamos valores faltantes y codificamos la variable objetivo para modelado.


In [ ]:
# Paso 3: Limpieza y preparación inicial
df_clean = df.copy()

# TotalCharges suele venir como texto por espacios vacíos en el dataset original
if 'TotalCharges' in df_clean.columns:
    df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Estandarización simple de campos categóricos tipo texto
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df_clean[col] = df_clean[col].replace({'': np.nan})

# Codificación de target
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

# Eliminamos filas donde el target no esté disponible
df_clean = df_clean.dropna(subset=['Churn']).copy()
df_clean['Churn'] = df_clean['Churn'].astype(int)

# Tratamiento de nulos para EDA y consistencia visual
numeric_cols_for_fill = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols_for_fill = df_clean.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_cols_for_fill:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols_for_fill:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('Limpieza completada.')
df_clean.head()


# 📊 4. Exploratory Analysis

Visualizaciones enfocadas en negocio para entender qué perfiles presentan mayor riesgo de churn.


In [ ]:
# Paso 4.1: Distribución de churn
ax = sns.countplot(data=df_clean, x='Churn', palette='Blues')
ax.set_title('Distribución de Customer Churn')
ax.set_xlabel('Churn (0 = No, 1 = Yes)')
ax.set_ylabel('Número de clientes')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()


In [ ]:
# Paso 4.2: Churn vs tipo de contrato
contract_churn = pd.crosstab(df_clean['Contract'], df_clean['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', stacked=True, color=['#9ecae1', '#08519c'])
plt.title('Churn por Tipo de Contrato')
plt.xlabel('Tipo de contrato')
plt.ylabel('Porcentaje de clientes')
plt.legend(['No Churn', 'Churn'], title='Resultado')
plt.tight_layout()
plt.show()


In [ ]:
# Paso 4.3: Churn vs MonthlyCharges
ax = sns.boxplot(data=df_clean, x='Churn', y='MonthlyCharges', palette='Blues')
ax.set_title('Monthly Charges según Churn')
ax.set_xlabel('Churn (0 = No, 1 = Yes)')
ax.set_ylabel('Cargo mensual')
plt.tight_layout()
plt.show()


In [ ]:
# Paso 4.4: Churn vs tenure
ax = sns.boxplot(data=df_clean, x='Churn', y='tenure', palette='Blues')
ax.set_title('Tenure según Churn')
ax.set_xlabel('Churn (0 = No, 1 = Yes)')
ax.set_ylabel('Meses de antigüedad')
plt.tight_layout()
plt.show()


In [ ]:
# Paso 4.5: Heatmap de correlación numérica
numeric_df = df_clean.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Mapa de Correlación de Variables Numéricas')
plt.tight_layout()
plt.show()


# 5. Feature Engineering

Separamos variables predictoras y objetivo, y preparamos una transformación sencilla para columnas numéricas y categóricas.


In [ ]:
# Paso 5: Preparación de features
model_df = df_clean.copy()

if 'customerID' in model_df.columns:
    model_df = model_df.drop(columns=['customerID'])

X = model_df.drop(columns=['Churn'])
y = model_df['Churn']

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Tamaño train: {X_train.shape}')
print(f'Tamaño test: {X_test.shape}')


# 🤖 6. Model Training

Entrenamos dos modelos base y registramos parámetros, accuracy y artefactos en MLflow para facilitar comparación y trazabilidad.


In [ ]:
# Paso 6: Entrenamiento con MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

model_results = []
trained_pipelines = {}

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])

    with mlflow.start_run(run_name=model_name):
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        params_to_log = {f'model__{k}': v for k, v in estimator.get_params().items()}
        mlflow.log_params(params_to_log)
        mlflow.log_metric('accuracy', accuracy)
        mlflow.sklearn.log_model(pipeline, artifact_path='model')

        model_results.append({
            'model_name': model_name,
            'accuracy': accuracy
        })
        trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(model_results).sort_values(by='accuracy', ascending=False).reset_index(drop=True)
results_df


# 7. Comparación de Modelos

Comparamos performance para seleccionar el mejor modelo del demo.


In [ ]:
# Paso 7: Selección de mejor modelo
print('Comparación de modelos:')
display(results_df)

best_model_name = results_df.loc[0, 'model_name']
best_model = trained_pipelines[best_model_name]

print(f'Mejor modelo seleccionado: {best_model_name}')

best_predictions = best_model.predict(X_test)
print('
Reporte de clasificación del mejor modelo:')
print(classification_report(y_test, best_predictions))

cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de Confusión - {best_model_name}')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()


# 8. Importancia de Variables

Para negocio, esta visualización ayuda a entender qué variables influyen más en el modelo de Random Forest.


In [ ]:
# Paso 8: Feature importance para Random Forest
rf_pipeline = trained_pipelines['RandomForestClassifier']
rf_model = rf_pipeline.named_steps['model']
rf_preprocessor = rf_pipeline.named_steps['preprocessor']

feature_names = rf_preprocessor.get_feature_names_out()
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False).head(10)

sns.barplot(data=importances, x='importance', y='feature', palette='Blues_r')
plt.title('Top 10 Variables Más Importantes - Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

importances


# 🔮 9. Prediction Demo

Creamos una función reutilizable para simular scoring de un cliente individual en un escenario de negocio.


In [ ]:
# Paso 9: Función de predicción para demo
training_columns = X.columns.tolist()

def predict_customer_churn(input_dict):
    \"\"\"Predice churn para un cliente individual usando el mejor modelo entrenado.\"\"\"
    input_df = pd.DataFrame([input_dict])

    for col in training_columns:
        if col not in input_df.columns:
            input_df[col] = np.nan

    input_df = input_df[training_columns].copy()

    if 'TotalCharges' in input_df.columns:
        input_df['TotalCharges'] = pd.to_numeric(input_df['TotalCharges'], errors='coerce')

    for col in input_df.select_dtypes(include='object').columns:
        input_df[col] = input_df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        input_df[col] = input_df[col].replace({'': np.nan})

    prediction = int(best_model.predict(input_df)[0])
    probability = float(best_model.predict_proba(input_df)[0][1])

    return {
        'prediction': prediction,
        'probability': probability
    }


# 10. Input Interactivo de Demo

Probamos la función con un cliente de ejemplo para mostrar el resultado final de cara a negocio o cliente.


In [ ]:
# Paso 10: Cliente de ejemplo y scoring
sample_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'Yes',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 89.5,
    'TotalCharges': 1074.0
}

demo_result = predict_customer_churn(sample_customer)
demo_probability_pct = demo_result['probability'] * 100
demo_label = 'Will Churn' if demo_result['prediction'] == 1 else 'Will Stay'

print(f'Customer churn probability: {demo_probability_pct:.2f}%')
print(f'Prediction: {demo_label}')


# 11. Instrucciones de MLflow UI

Para abrir la interfaz de MLflow en el servidor, ejecuta:

```bash
mlflow ui --host 0.0.0.0 --port 5000
```

## Acceso desde navegador en DigitalOcean
- Usa la IP pública del droplet.
- Abre en tu navegador: `http://TU_IP_PUBLICA:5000`
- Verifica que el puerto `5000` esté habilitado en el firewall del droplet.
- Si usas un proxy reverso o reglas de red, asegúrate de exponer ese puerto correctamente.

Con esto podrás revisar corridas, métricas, parámetros y modelos registrados durante la demo.
